### Importing Packages and Loading Data

In [1]:
import pandas as pd
import os
from email.parser import Parser

In [2]:
POI = ['lay-k', 'skilling-j']
folders = ['sent', 'deleted_items', 'discussion_threads', 'notes_inbox']
mdir = os.path.join('..', 'Enron_Investigation') 
email_data = []

In [3]:
for person in POI: #for each person in the list POI
    person_path = os.path.join(mdir, person) #create a single path from the main repo to the persons in POI
    for folder in os.listdir(person_path): #create a list of the subfolder for each person
        if any(key in folder.lower() for key in folders): #if any word in a person's folder matches a word in folders
            folder_path = os.path.join(person_path, folder) #look at every file inside the folders that match the words i folders
            for file in os.listdir(folder_path): #for every file in folder path create a list of files
                file_path = os.path.join(folder_path, file) #join the path for the folder with the file
                if os.path.isfile(file_path): #check to make sure each item in file path is a file
                    with open(file_path, 'r', errors='ignore') as f: #Open and read every file, regardless of errors
                        msg = Parser().parsestr(f.read()) #decodes the text into an email format
                        email_data.append({ #creates a dictionary of the fields
                            'date': msg['Date'], 
                            'from': msg['From'],
                            'to': msg['To'], 
                            'subject': msg['Subject'], 
                            'body': msg.get_payload(), 
                            'custodian': person, 
                            'folder_source': folder
                        })
df = pd.DataFrame(email_data) #makes the dictionary into a dataframe
print(f"Success! Captured {len(df)} high-value emails.") 

Success! Captured 5359 high-value emails.


### Data Overview

In [4]:
df.head()

,date,from,to,subject,body,custodian,folder_source
0,"Wed, 30 Jan 2002 11:06:10 -0800 (PST)",grnthumb@ipa.netf,klay@enron.com,Demand Ken Lay Donate Proceeds from Enron Stoc...,"Linda Brunner\n12335 S. 2225 Rd.\nStockton, MO...",lay-k,deleted_items
1,"Wed, 30 Jan 2002 11:39:03 -0800 (PST)",avanleunen@msn.com,klay@enron.com,Demand Ken Lay Donate Proceeds from Enron Stoc...,"alice van leunen\n13631 NE Kinney Rd\nNewberg,...",lay-k,deleted_items
2,"Wed, 30 Jan 2002 11:42:03 -0800 (PST)",comet@starshaper.com,klay@enron.com,Demand Ken Lay Donate Proceeds from Enron Stoc...,John-Paul Goorjian\ncomet@starshaper.com\n\nTo...,lay-k,deleted_items
3,"Wed, 30 Jan 2002 09:57:08 -0800 (PST)",manastassatos@randomhouse.com,klay@enron.com,Demand Ken Lay Donate Proceeds from Enron Stoc...,Marietta Anastassatos\n435 Highland Ave\nPalis...,lay-k,deleted_items
4,"Wed, 30 Jan 2002 09:57:10 -0800 (PST)",kmitchel@coba.usf.edu,klay@enron.com,Demand Ken Lay Donate Proceeds from Enron Stoc...,Kimberly Mitchell\n1906 E. Linda Street\nPlant...,lay-k,deleted_items


In [5]:
df.tail()

,date,from,to,subject,body,custodian,folder_source
5354,"Wed, 12 Apr 2000 04:40:00 -0700 (PDT)",sherri.sera@enron.com,maktay@superonline.com,Hi...,"Metin,\n\nGreat to hear from you. Believe it ...",skilling-j,_sent_mail
5355,"Mon, 10 Apr 2000 10:52:00 -0700 (PDT)",sherri.sera@enron.com,annb@redstonegrp.com,RE: Message from Gary Stein,Ugh! :-) Just kidding...I figured it would be ...,skilling-j,_sent_mail
5356,"Mon, 10 Apr 2000 10:38:00 -0700 (PDT)",sherri.sera@enron.com,annb@redstonegrp.com,RE: Message from Gary Stein,I shall be there. Just let me know what time ...,skilling-j,_sent_mail
5357,"Mon, 10 Apr 2000 10:05:00 -0700 (PDT)",sherri.sera@enron.com,annb@redstonegrp.com,Re: Message from Gary Stein,"Ann, I haven't heard from you with regard to w...",skilling-j,_sent_mail
5358,"Fri, 7 Apr 2000 10:11:00 -0700 (PDT)",sherri.sera@enron.com,chuck.paul@a-closer-look.com,Re: Fw:,"Dear Mr. Paul,\n\nI'm writing on behalf of Jef...",skilling-j,_sent_mail


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5359 entries, 0 to 5358
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   date           5359 non-null   object
 1   from           5359 non-null   object
 2   to             5104 non-null   object
 3   subject        5359 non-null   object
 4   body           5359 non-null   object
 5   custodian      5359 non-null   object
 6   folder_source  5359 non-null   object
dtypes: object(7)
memory usage: 293.2+ KB


In [7]:
df['custodian'].value_counts()

custodian
lay-k         3375
skilling-j    1984
Name: count, dtype: int64

In [8]:
df[df['custodian'] == 'skilling-j']['from'].value_counts().head(10)

from
sherri.sera@enron.com            466
joannie.williamson@enron.com      99
jeff.skilling@enron.com           87
40enron@enron.com                 70
markskilling@hotmail.com          51
fredinvt@juno.com                 41
sherri.reinartz@enron.com         36
katherine.brown@enron.com         31
lisa.connolly@enron.com           26
enron.announcements@enron.com     25
Name: count, dtype: int64

In [9]:
df[df['custodian'] == 'lay-k']['from'].value_counts().head(10)

from
rosalee.fleming@enron.com        595
leonardo.pacheco@enron.com        52
brown_mary_jo@lilly.com           46
savont@email.msn.com              44
tori.wells@enron.com              41
katherine.brown@enron.com         30
elizabeth.davis@compaq.com        29
rob.bradley@enron.com             20
enron.announcements@enron.com     20
lizard_ar@yahoo.com               18
Name: count, dtype: int64

### Converting date column to datetime

In [10]:
df['date'].head()

0    Wed, 30 Jan 2002 11:06:10 -0800 (PST)
1    Wed, 30 Jan 2002 11:39:03 -0800 (PST)
2    Wed, 30 Jan 2002 11:42:03 -0800 (PST)
3    Wed, 30 Jan 2002 09:57:08 -0800 (PST)
4    Wed, 30 Jan 2002 09:57:10 -0800 (PST)
Name: date, dtype: object

In [11]:
df['date_clean'] = pd.to_datetime(df['date'], utc=True)

C:\Users\AlexaJohnston\AppData\Local\Temp\ipykernel_40920\1452036759.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date_clean'] = pd.to_datetime(df['date'], utc=True)


In [12]:
df = df[(df['date_clean'].dt.year >= 1998) & (df['date_clean'].dt.year <= 2001)]

In [13]:
df.head()

,date,from,to,subject,body,custodian,folder_source,date_clean
156,"Tue, 13 Nov 2001 11:14:52 -0800 (PST)",ohi_supporters@operationhope.org,klay@enron.com,The Latest Edition of Operation HOPE Inc.'s El...,\n \nHere It Is !!! \n[IMAGE]\nor copy and pa...,lay-k,deleted_items,2001-11-13 19:14:52+00:00
158,"Mon, 26 Nov 2001 17:37:47 -0800 (PST)",djtheroux@independent.org,lighthouse@independent.org,"THE LIGHTHOUSE: November 26, 2001","THE LIGHTHOUSE\n""Enlightening Ideas for Public...",lay-k,deleted_items,2001-11-27 01:37:47+00:00
1126,"Mon, 2 Aug 1999 08:33:00 -0700 (PDT)",ggalata@enron.co.uk,kenneth.lay@enron.com,Franco Bernabe's tel number,"Ken,\n\nI finally found Franco Bernabe's work ...",lay-k,discussion_threads,1999-08-02 15:33:00+00:00
1127,"Thu, 19 Aug 1999 02:18:00 -0700 (PDT)",rosalee.fleming@enron.com,expense.report@enron.com,Approval of Jeff Skilling's Expense Report,Kenneth Lay has approved the attached expense ...,lay-k,discussion_threads,1999-08-19 09:18:00+00:00
1128,"Mon, 21 Aug 2000 02:36:00 -0700 (PDT)",corry.bentley@enron.com,kenneth.lay@enron.com,Sacrifice,"Here in my group, and all over Enron, I see pe...",lay-k,discussion_threads,2000-08-21 09:36:00+00:00


In [14]:
df.tail()

,date,from,to,subject,body,custodian,folder_source,date_clean
5354,"Wed, 12 Apr 2000 04:40:00 -0700 (PDT)",sherri.sera@enron.com,maktay@superonline.com,Hi...,"Metin,\n\nGreat to hear from you. Believe it ...",skilling-j,_sent_mail,2000-04-12 11:40:00+00:00
5355,"Mon, 10 Apr 2000 10:52:00 -0700 (PDT)",sherri.sera@enron.com,annb@redstonegrp.com,RE: Message from Gary Stein,Ugh! :-) Just kidding...I figured it would be ...,skilling-j,_sent_mail,2000-04-10 17:52:00+00:00
5356,"Mon, 10 Apr 2000 10:38:00 -0700 (PDT)",sherri.sera@enron.com,annb@redstonegrp.com,RE: Message from Gary Stein,I shall be there. Just let me know what time ...,skilling-j,_sent_mail,2000-04-10 17:38:00+00:00
5357,"Mon, 10 Apr 2000 10:05:00 -0700 (PDT)",sherri.sera@enron.com,annb@redstonegrp.com,Re: Message from Gary Stein,"Ann, I haven't heard from you with regard to w...",skilling-j,_sent_mail,2000-04-10 17:05:00+00:00
5358,"Fri, 7 Apr 2000 10:11:00 -0700 (PDT)",sherri.sera@enron.com,chuck.paul@a-closer-look.com,Re: Fw:,"Dear Mr. Paul,\n\nI'm writing on behalf of Jef...",skilling-j,_sent_mail,2000-04-07 17:11:00+00:00


### Handling duplicates

In [15]:
count = len(df) 
df = df.drop_duplicates(subset=['date', 'from', 'subject', 'body'])
count_2 = len(df) 
print(f"Removed {count - count_2} duplicate emails.") 

Removed 1754 duplicate emails.


### Handling missing data

In [16]:
df['to'].isnull().sum()

np.int64(186)

In [17]:
missing_to = df[df['to'].isnull()]
print(missing_to['folder_source'].value_counts())

folder_source
deleted_items         108
discussion_threads     73
notes_inbox             5
Name: count, dtype: int64


In [18]:
import numpy as np 
df['to'] = df['to'].replace(r'^\s*$', np.nan, regex=True)
df['to'] = df['to'].fillna('Unknown Recipient') 

### Summary

In [19]:
print(f"Unique Emails: {len(df)}")
print(f"Unknown Recipients: {(df['to'] == 'Unknown Recipient').sum()}")

Unique Emails: 2476
Unknown Recipients: 186


In [20]:
df.to_csv(r'C:\Users\AlexaJohnston\Springboard\Enron_Investigation_Capstone\cleaned_dataset.csv', index=False)